# 02 - CUDA 手写

> 配套笔记：[../notes/02_算子手写(1) - CUDA 入门 + 常见 op.md](../notes/02_算子手写(1)%20-%20CUDA%20入门%20+%20常见%20op.md) ｜ 知乎：<https://zhuanlan.zhihu.com/p/1892487783110644443>

按面试常考频率覆盖 8 类算子：
1. **Add** — vecAdd（含 Float4）/ matAdd
2. **Matmul** — naive / Tiled
3. **Transpose**
4. **Reduce 1D** — atomic / block 折半 / warp shuffle / warp + Float4
5. **Reduce 2D** — Col / Row
6. **Softmax** — naive / Float4
7. **Conv** — 1D/2D × naive/Tiled
8. **LayerNorm**

每个 kernel 都附 test case，可在 Colab 直接 run。

## 0. 准备

写 kernel 的「3 步走」：
- **宏定义**：`CEIL`、`BLOCKSIZE`、`FLOAT4`（向量化访存）
- **调用函数**：定义 `threadPerBlock` / `blockPerGrid`，用 `<<<...>>>` 启动 + `cudaDeviceSynchronize()`
- **kernel 函数**：定义每个 thread 做什么

**Float4 合并访存**：用 `LDG.128` 一次读 4 个 float（PTX 最大就是 128 bit），减少指令数。

> ⚠️ Colab 选 GPU runtime（T4 即可），否则 `nvidia-smi` / `which nvcc` 没输出。

In [ ]:
!nvidia-smi

In [ ]:
!which nvcc
# 没有输出意味着环境有问题

## 01 - Add

考察频率：中。

每个 thread 算 1 个元素（或 4 个，Float4 版）。重点是 index 的计算：
- 1D：`idx = bid.x * bdim.x + tid.x`
- 2D：`row = bid.y * bdim.y + tid.y`，`col = bid.x * bdim.x + tid.x`

### 1D vecAdd

最基础版：`BLOCKSIZE = 1024`，每 thread 处理 1 个 float。

In [ ]:
%%writefile vecAdd.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define BLOCKSIZE 1024
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void vecAddKernel(const float *A, const float *B, float *C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        C[idx] = A[idx] + B[idx];
    }
}

void vecAdd(const float *A, const float *B, float *C, int N) {
    dim3 threadPerBlock(BLOCKSIZE);
    dim3 blockPerGrid(CEIL(N, BLOCKSIZE));
    vecAddKernel<<<blockPerGrid, threadPerBlock>>>(A, B, C, N);
    cudaDeviceSynchronize();
}

// 简化的测试函数（仅测试 N=5）
void simple_test() {
    const int N = 5;
    float h_A[N] = {1.0, 2.0, 3.0, 4.0, 5.0};  // 固定输入 A
    float h_B[N] = {0.1, 0.2, 0.3, 0.4, 0.5};  // 固定输入 B
    float h_C_gpu[N] = {0};
    float h_C_cpu[N] = {0};

    // --- GPU 计算 ---
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, N * sizeof(float));
    cudaMalloc(&d_B, N * sizeof(float));
    cudaMalloc(&d_C, N * sizeof(float));

    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(float), cudaMemcpyHostToDevice);

    vecAdd(d_A, d_B, d_C, N);  // 调用 GPU 函数

    cudaMemcpy(h_C_gpu, d_C, N * sizeof(float), cudaMemcpyDeviceToHost);

    // --- CPU 计算（参考结果）---
    for (int i = 0; i < N; i++) {
        h_C_cpu[i] = h_A[i] + h_B[i];
    }

    // --- 打印输入输出 ---
    printf("Input A: ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_A[i]);
    printf("\nInput B: ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_B[i]);
    printf("\nGPU Result: ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_C_gpu[i]);
    printf("\nCPU Result: ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_C_cpu[i]);
    printf("\n");

    // --- 简单验证 ---
    bool pass = true;
    const float eps = 1e-5;
    for (int i = 0; i < N; i++) {
        if (fabs(h_C_gpu[i] - h_C_cpu[i]) > eps) {
            pass = false;
            break;
        }
    }
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 清理资源
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

int main() {
    simple_test();
    return 0;
}

In [ ]:
!nvcc vecAdd.cu -o vecAdd -arch=sm_75 && ./vecAdd

### 1D vecAdd + Float4

每 thread 处理 4 个元素：
- index 起点 ×4
- 用 `FLOAT4(A[idx])` 宏把地址重解释为 `float4*`
- `blockPerGrid` 缩小 4 倍

In [ ]:
%%writefile vecAddV4.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define BLOCKSIZE 1024
#define CEIL(a, b) ((a + b - 1) / b)
#define FLOAT4(value) (reinterpret_cast<float4*>(&(value))[0])  // 定义 FLOAT4 宏

__global__ void vecAddKernel(float *A, float *B, float *C, int N){
  int idx = (blockIdx.x * blockDim.x + threadIdx.x) * 4;
  if (idx < N){
    float4 tmp_A = FLOAT4(A[idx]);
    float4 tmp_B = FLOAT4(B[idx]);
    float4 tmp_C;
    tmp_C.x = tmp_A.x + tmp_B.x;
    tmp_C.y = tmp_A.y + tmp_B.y;
    tmp_C.z = tmp_A.z + tmp_B.z;
    tmp_C.w = tmp_A.w + tmp_B.w;
    FLOAT4(C[idx]) = tmp_C;
  }
}

void vecAdd(float *A, float *B, float *C, int N){
  dim3 threadPerBlock = BLOCKSIZE;
  dim3 blockPerGrid = CEIL(CEIL(N,4), BLOCKSIZE);	// modify

  vecAddKernel<<<blockPerGrid, threadPerBlock>>>(A, B, C, N);
  cudaDeviceSynchronize();
}

// 简化的测试函数（测试 N=8，确保对齐）
void simple_test() {
    const int N = 8;
    float h_A[N] = {1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0};
    float h_B[N] = {0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8};
    float h_C_gpu[N] = {0};
    float h_C_cpu[N] = {0};

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, N * sizeof(float));
    cudaMalloc(&d_B, N * sizeof(float));
    cudaMalloc(&d_C, N * sizeof(float));

    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(float), cudaMemcpyHostToDevice);

    vecAdd(d_A, d_B, d_C, N);

    cudaMemcpy(h_C_gpu, d_C, N * sizeof(float), cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++) {
        h_C_cpu[i] = h_A[i] + h_B[i];
    }

    printf("Input A: ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_A[i]);
    printf("\nInput B: ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_B[i]);
    printf("\nGPU Result: ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_C_gpu[i]);
    printf("\nCPU Result: ");
    for (int i = 0; i < N; i++) printf("%.1f ", h_C_cpu[i]);

    bool pass = true;
    const float eps = 1e-5;
    for (int i = 0; i < N; i++) {
        if (fabs(h_C_gpu[i] - h_C_cpu[i]) > eps) {
            pass = false;
            break;
        }
    }
    printf("\n\nTest: %s\n", pass ? "PASS" : "FAIL");

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

int main() {
    simple_test();
    return 0;
}


In [ ]:
!nvcc vecAddV4.cu -o vecAddV4 -arch=sm_75 && ./vecAddV4

### 2D matAdd

逻辑同 1D，只是 index 用 `row * N + col` 算线性偏移。

In [ ]:
%%writefile matAdd2D.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cstdlib>  // 用于 rand()

#define BLOCKSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

// 你的核函数（保持原样）
__global__ void MatAddKernel(float *A, float *B, float *C, int M, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < M && col < N) {
        int idx = row * N + col;
        C[idx] = A[idx] + B[idx];
    }
}

// 宿主函数（修正语法错误）
void MatAdd(float *A, float *B, float *C, int M, int N) {
    dim3 threadPerBlock(BLOCKSIZE, BLOCKSIZE);  // 修正为 dim3 构造函数
    dim3 blockPerGrid(CEIL(N, BLOCKSIZE), CEIL(M, BLOCKSIZE));
    MatAddKernel<<<blockPerGrid, threadPerBlock>>>(A, B, C, M, N);
    cudaDeviceSynchronize();
}

// 测试函数（验证 M=3, N=5 的小矩阵）
void test_MatAdd() {
    const int M = 3;
    const int N = 5;
    const int total = M * N;

    // 初始化主机数据
    float h_A[total], h_B[total], h_C_gpu[total], h_C_cpu[total];
    for (int i = 0; i < total; i++) {
        h_A[i] = i + 1.0f;       // A = [[1,2,3,4,5], [6,7,8,9,10], [11,12,13,14,15]]
        h_B[i] = i * 0.1f;       // B = [[0.0,0.1,0.2,0.3,0.4], [0.5,0.6,...], ...]
    }

    // 分配设备内存
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, total * sizeof(float));
    cudaMalloc(&d_B, total * sizeof(float));
    cudaMalloc(&d_C, total * sizeof(float));

    // 拷贝数据到设备
    cudaMemcpy(d_A, h_A, total * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, total * sizeof(float), cudaMemcpyHostToDevice);

    // 执行 GPU 计算
    MatAdd(d_A, d_B, d_C, M, N);

    // 拷贝结果回主机
    cudaMemcpy(h_C_gpu, d_C, total * sizeof(float), cudaMemcpyDeviceToHost);

    // CPU 计算参考结果
    for (int i = 0; i < total; i++) {
        h_C_cpu[i] = h_A[i] + h_B[i];
    }

    // 打印结果 (仅打印前两行示例)
    printf("Matrix A (First 2 rows):\n");
    for (int i = 0; i < 2; i++) {
        for (int j = 0; j < N; j++) printf("%5.1f", h_A[i * N + j]);
        printf("\n");
    }
    printf("\nMatrix B (First 2 rows):\n");
    for (int i = 0; i < 2; i++) {
        for (int j = 0; j < N; j++) printf("%5.1f", h_B[i * N + j]);
        printf("\n");
    }
    printf("\nGPU Result (First 2 rows):\n");
    for (int i = 0; i < 2; i++) {
        for (int j = 0; j < N; j++) printf("%5.1f", h_C_gpu[i * N + j]);
        printf("\n");
    }

    // 验证结果
    bool pass = true;
    const float eps = 1e-5;
    for (int i = 0; i < total; i++) {
        if (fabs(h_C_gpu[i] - h_C_cpu[i]) > eps) {
            pass = false;
            printf("Error at index %d: GPU=%.2f, CPU=%.2f\n", i, h_C_gpu[i], h_C_cpu[i]);
            break;
        }
    }
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 释放内存
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

int main() {
    test_MatAdd();
    return 0;
}

In [ ]:
!nvcc matAdd2D.cu -o matAdd2D -arch=sm_75 && ./matAdd2D

## 02 - Matmul 矩阵乘

考察频率：超级高。被问直接上 Tiled 版本。

复杂度对比（Tile 边长 $d$）：
- Naive：load/store $\sim 2mnp$
- Tiled：load/store $\sim 2mnp/d$（差一个 $d$ 倍）

### Naive Matmul

每个 thread 算 C 里的一个元素，循环 K 次累加 `A[row, i] * B[i, col]`。

In [ ]:
%%writefile matMul2D.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cstdlib>  // 用于 rand()

#define BLOCKSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void matMulKernel(float *A, float *B, float *C, int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < M && col < N) {
        float sum = 0.0f;
        for (int i = 0; i < K; i++) {
            sum += A[row * K + i] * B[i * N + col];
        }
        C[row * N + col] = sum;
    }
}

void matMul(float *A, float *B, float *C, int M, int N, int K) {
    dim3 threadPerBlock(BLOCKSIZE, BLOCKSIZE);
    dim3 blockPerGrid(CEIL(N, BLOCKSIZE), CEIL(M, BLOCKSIZE));
    matMulKernel<<<blockPerGrid, threadPerBlock>>>(A, B, C, M, N, K);
    cudaDeviceSynchronize();
}

// 测试函数（验证 M=2, N=3, K=4 的小矩阵）
void test_MatMul() {
    const int M = 2, N = 3, K = 4;
    const int size_A = M * K, size_B = K * N, size_C = M * N;

    // 初始化主机数据（示例数据）
    float h_A[size_A] = {1.0, 2.0, 3.0, 4.0,  // A = [[1,2,3,4],
                        5.0, 6.0, 7.0, 8.0};  //      [5,6,7,8]]
    float h_B[size_B] = {1.0, 0.0, 0.0,       // B = [[1,0,0],
                        0.0, 1.0, 0.0,        //      [0,1,0],
                        0.0, 0.0, 1.0,        //      [0,0,1],
                        0.0, 0.0, 0.0};       //      [0,0,0]]
    float h_C_gpu[size_C] = {0};
    float h_C_cpu[size_C] = {0};

    // 分配设备内存
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size_A * sizeof(float));
    cudaMalloc(&d_B, size_B * sizeof(float));
    cudaMalloc(&d_C, size_C * sizeof(float));

    // 拷贝数据到设备
    cudaMemcpy(d_A, h_A, size_A * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B * sizeof(float), cudaMemcpyHostToDevice);

    // 执行 GPU 计算
    matMul(d_A, d_B, d_C, M, N, K);

    // 拷贝结果回主机
    cudaMemcpy(h_C_gpu, d_C, size_C * sizeof(float), cudaMemcpyDeviceToHost);

    // CPU 计算参考结果
    for (int row = 0; row < M; row++) {
        for (int col = 0; col < N; col++) {
            float sum = 0.0f;
            for (int i = 0; i < K; i++) {
                sum += h_A[row * K + i] * h_B[i * N + col];
            }
            h_C_cpu[row * N + col] = sum;
        }
    }

    // 打印输入输出（仅打印完整矩阵）
    printf("Matrix A (%dx%d):\n", M, K);
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < K; j++) printf("%5.1f", h_A[i * K + j]);
        printf("\n");
    }
    printf("\nMatrix B (%dx%d):\n", K, N);
    for (int i = 0; i < K; i++) {
        for (int j = 0; j < N; j++) printf("%5.1f", h_B[i * N + j]);
        printf("\n");
    }
    printf("\nGPU Result (%dx%d):\n", M, N);
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) printf("%5.1f", h_C_gpu[i * N + j]);
        printf("\n");
    }

    // 验证结果
    bool pass = true;
    const float eps = 1e-5;
    for (int i = 0; i < size_C; i++) {
        if (fabs(h_C_gpu[i] - h_C_cpu[i]) > eps) {
            pass = false;
            printf("Error at index %d: GPU=%.2f, CPU=%.2f\n", i, h_C_gpu[i], h_C_cpu[i]);
            break;
        }
    }
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 释放内存
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

int main() {
    test_MatMul();
    return 0;
}

In [ ]:
!nvcc matMul2D.cu -o matMul2D -arch=sm_75 && ./matMul2D

### Tiled Matmul

核心思路：用 SMEM 缓存子矩阵，减少 global memory 访问。
- 每个 block 算 C 的一个 `BLOCKSIZE × BLOCKSIZE` 子矩阵
- 沿 K 维分 `numTiles` 次：load A/B 子块到 SMEM → `__syncthreads()` → 累加局部矩乘 → 再 sync
- 第二次 sync 是为了让 SMEM 在下一轮被覆盖前所有 thread 都用完

In [ ]:
%%writefile matMulTiled.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cstdlib>

#define BLOCKSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void matMulTiledKernel(float *A, float *B, float *C, int M, int N, int K) {
    __shared__ float As[BLOCKSIZE][BLOCKSIZE];
    __shared__ float Bs[BLOCKSIZE][BLOCKSIZE];

    int row = blockIdx.y * BLOCKSIZE + threadIdx.y;
    int col = blockIdx.x * BLOCKSIZE + threadIdx.x;
    float sum = 0.0f;
    int numTiles = CEIL(K, BLOCKSIZE);

    for (int t = 0; t < numTiles; t++) {
        int offset = t * BLOCKSIZE;

        // 加载分块 A
        int loadA_col = offset + threadIdx.x;
        if (row < M && loadA_col < K) {
            As[threadIdx.y][threadIdx.x] = A[row * K + loadA_col];
        } else {
            As[threadIdx.y][threadIdx.x] = 0.0f;
        }

        // 加载分块 B
        int loadB_row = offset + threadIdx.y;
        if (col < N && loadB_row < K) {
            Bs[threadIdx.y][threadIdx.x] = B[loadB_row * N + col];
        } else {
            Bs[threadIdx.y][threadIdx.x] = 0.0f;
        }

        __syncthreads();

        // 计算分块乘积累加
        for (int k = 0; k < BLOCKSIZE; k++) {
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        }
        __syncthreads();
    }

    // 存储结果
    if (row < M && col < N) {
        C[row * N + col] = sum;
    }
}

void matMul(float *A, float *B, float *C, int M, int N, int K) {
    dim3 threadPerBlock(BLOCKSIZE, BLOCKSIZE);
    dim3 blockPerGrid(CEIL(N, BLOCKSIZE), CEIL(M, BLOCKSIZE));
    matMulTiledKernel<<<blockPerGrid, threadPerBlock>>>(A, B, C, M, N, K);
    cudaDeviceSynchronize();
}

// 测试函数（验证 M=2, N=3, K=4）
void test_MatMul() {
    const int M = 2, N = 3, K = 4;
    const int size_A = M * K, size_B = K * N, size_C = M * N;

    // 初始化主机数据（手工可验证）
    float h_A[size_A] = {1.0, 2.0, 3.0, 4.0,  // A = [[1,2,3,4],
                        5.0, 6.0, 7.0, 8.0};  //      [5,6,7,8]]
    float h_B[size_B] = {1.0, 0.0, 0.0,       // B = [[1,0,0],
                        0.0, 1.0, 0.0,        //      [0,1,0],
                        0.0, 0.0, 1.0,        //      [0,0,1],
                        0.0, 0.0, 0.0};       //      [0,0,0]]
    float h_C_gpu[size_C] = {0};
    float h_C_cpu[size_C] = {0};

    // CPU 计算参考结果
    for (int row = 0; row < M; row++) {
        for (int col = 0; col < N; col++) {
            float sum = 0.0f;
            for (int i = 0; i < K; i++) {
                sum += h_A[row * K + i] * h_B[i * N + col];
            }
            h_C_cpu[row * N + col] = sum;
        }
    }

    // GPU 计算
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size_A * sizeof(float));
    cudaMalloc(&d_B, size_B * sizeof(float));
    cudaMalloc(&d_C, size_C * sizeof(float));

    cudaMemcpy(d_A, h_A, size_A * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B * sizeof(float), cudaMemcpyHostToDevice);

    matMul(d_A, d_B, d_C, M, N, K);
    cudaMemcpy(h_C_gpu, d_C, size_C * sizeof(float), cudaMemcpyDeviceToHost);

    // 打印结果
    printf("Matrix A (%dx%d):\n", M, K);
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < K; j++) printf("%5.1f", h_A[i * K + j]);
        printf("\n");
    }
    printf("\nMatrix B (%dx%d):\n", K, N);
    for (int i = 0; i < K; i++) {
        for (int j = 0; j < N; j++) printf("%5.1f", h_B[i * N + j]);
        printf("\n");
    }
    printf("\nGPU Result (%dx%d):\n", M, N);
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) printf("%5.1f", h_C_gpu[i * N + j]);
        printf("\n");
    }

    // 验证正确性
    bool pass = true;
    const float eps = 1e-5;
    for (int i = 0; i < size_C; i++) {
        if (fabs(h_C_gpu[i] - h_C_cpu[i]) > eps) {
            pass = false;
            printf("Error at index %d: GPU=%.2f, CPU=%.2f\n", i, h_C_gpu[i], h_C_cpu[i]);
            break;
        }
    }
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 清理资源
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

int main() {
    test_MatMul();
    return 0;
}

In [ ]:
!nvcc matMulTiled.cu -o matMulTiled -arch=sm_75 && ./matMulTiled

## 03 - Transpose

考察频率：中。

非常简单：`B[col * M + row] = A[row * N + col]`。

In [ ]:
%%writefile matTranspose.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define BLOCKSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void matTransposeKernel(float *A, float *B, int M, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < M && col < N) {
        int idx_A = row * N + col;
        int idx_B = col * M + row;
        B[idx_B] = A[idx_A];
    }
}

void matTranspose(float *A, float *B, int M, int N) {
    dim3 threadPerBlock(BLOCKSIZE, BLOCKSIZE);
    dim3 blockPerGrid(CEIL(N, BLOCKSIZE), CEIL(M, BLOCKSIZE));
    matTransposeKernel<<<blockPerGrid, threadPerBlock>>>(A, B, M, N);
    cudaDeviceSynchronize();
}

// 测试函数（验证 M=3, N=4 的小矩阵）
void test_MatTranspose() {
    const int M = 3, N = 4;
    const int size_A = M * N, size_B = N * M;

    // 初始化输入矩阵 A（按行填充连续值）
    float h_A[size_A] = {1.0,  2.0,  3.0,  4.0,   // A = [[1, 2, 3, 4],
                        5.0,  6.0,  7.0,  8.0,   //      [5, 6, 7, 8],
                        9.0, 10.0, 11.0, 12.0};  //      [9,10,11,12]]
    float h_B_gpu[size_B] = {0};
    float h_B_cpu[size_B] = {0};

    // 分配设备内存
    float *d_A, *d_B;
    cudaMalloc(&d_A, size_A * sizeof(float));
    cudaMalloc(&d_B, size_B * sizeof(float));

    // 拷贝数据到设备
    cudaMemcpy(d_A, h_A, size_A * sizeof(float), cudaMemcpyHostToDevice);

    // 执行 GPU 转置
    matTranspose(d_A, d_B, M, N);

    // 拷贝结果回主机
    cudaMemcpy(h_B_gpu, d_B, size_B * sizeof(float), cudaMemcpyDeviceToHost);

    // CPU 计算参考结果
    for (int row = 0; row < M; row++) {
        for (int col = 0; col < N; col++) {
            int idx_A = row * N + col;
            int idx_B = col * M + row;
            h_B_cpu[idx_B] = h_A[idx_A];
        }
    }

    // 打印输入输出
    printf("Input Matrix A (%dx%d):\n", M, N);
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) printf("%4.1f ", h_A[i * N + j]);
        printf("\n");
    }
    printf("\nGPU Transposed Result (%dx%d):\n", N, M);
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < M; j++) printf("%4.1f ", h_B_gpu[i * M + j]);
        printf("\n");
    }

    // 验证结果
    bool pass = true;
    const float eps = 1e-5;
    for (int i = 0; i < size_B; i++) {
        if (fabs(h_B_gpu[i] - h_B_cpu[i]) > eps) {
            pass = false;
            printf("Error at index %d: GPU=%.2f, CPU=%.2f\n", i, h_B_gpu[i], h_B_cpu[i]);
            break;
        }
    }
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 释放内存
    cudaFree(d_A);
    cudaFree(d_B);
}

int main() {
    test_MatTranspose();
    return 0;
}

In [ ]:
!nvcc matTranspose.cu -o matTranspose -arch=sm_75 && ./matTranspose

## 04 - Reduce 1D（sum 规约）

考察频率：高。被问直接上 warp shuffle 版。

四种实现，性能 atomic < block 折半 < warp shuffle < warp + Float4。

### Naive - Atomic

每个 thread 直接 `atomicAdd` 到全局 output。原子操作不可并行 → 最慢。

In [ ]:
%%writefile reduceNaive.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cstdlib>  // 用于 rand()

#define BLOCKSIZE 1024
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void reduceKernelV1(const float *A, float *output, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        atomicAdd(output, A[idx]);
    }
}

void reduce(const float *A, float *output, int N) {
    dim3 threadPerBlock(BLOCKSIZE);
    dim3 blockPerGrid(CEIL(N, BLOCKSIZE));

    reduceKernelV1<<<blockPerGrid, threadPerBlock>>>(A, output, N);
    cudaDeviceSynchronize();
}

// 测试函数（验证 N=1024）
void test_reduce() {
    const int N = 1024;
    float h_A[N];
    float h_output_gpu = 0.0f;
    float h_output_cpu = 0.0f;

    // 初始化输入数据（随机值 [0,1)）
    srand(42);  // 固定随机种子
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)rand() / RAND_MAX;
        h_output_cpu += h_A[i];
    }

    // 分配设备内存
    float *d_A, *d_output;
    cudaMalloc(&d_A, N * sizeof(float));
    cudaMalloc(&d_output, sizeof(float));

    // 在测试函数中初始化输出内存为 0
    cudaMemset(d_output, 0, sizeof(float));

    // 拷贝数据到设备
    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);

    // 执行归约
    reduce(d_A, d_output, N);

    // 拷贝结果回主机
    cudaMemcpy(&h_output_gpu, d_output, sizeof(float), cudaMemcpyDeviceToHost);

    // 打印结果
    printf("CPU Sum: %.6f\n", h_output_cpu);
    printf("GPU Sum: %.6f\n", h_output_gpu);

    // 验证结果
    const float eps = 1e-3;
    bool pass = fabs(h_output_gpu - h_output_cpu) < eps;
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 释放内存
    cudaFree(d_A);
    cudaFree(d_output);
}

int main() {
    test_reduce();
    return 0;
}

In [ ]:
!nvcc reduceNaive.cu -o reduceNaive -arch=sm_75 && ./reduceNaive

### Block 内折半归约

- 先 load 到 SMEM
- 折半累加：每轮 offset 减半，前 offset 个 thread 把 `As[tid+offset]` 加到 `As[tid]`
- 每轮之间需要 `__syncthreads()`
- 要求 `BLOCKSIZE` 是 2 的幂

In [ ]:
%%writefile reduceBlock.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cstdlib>

#define BLOCKSIZE 1024
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void reduceBlockKernel(float *A, float *output, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int tid = threadIdx.x;
    __shared__ float As[BLOCKSIZE];

    // 加载到共享内存（带边界检查）
    As[tid] = (idx < N) ? A[idx] : 0;
    __syncthreads();

    // 折半归约
    for (int offset = BLOCKSIZE >> 1; offset > 0; offset >>= 1) {
        if (tid < offset) As[tid] += As[tid + offset];
        __syncthreads();
    }

    // 将块结果累加到全局内存
    if (tid == 0) {
        atomicAdd(output, As[0]);
    }
}

void reduce(float *A, float *output, int N) {
    dim3 threadPerBlock(BLOCKSIZE);
    dim3 blockPerGrid(CEIL(N, BLOCKSIZE));

    reduceBlockKernel<<<blockPerGrid, threadPerBlock>>>(A, output, N);
    cudaDeviceSynchronize();
}

// 测试函数（验证 N=10000）
void test_reduce() {
    const int N = 10000;
    float h_A[N];
    float h_output_gpu = 0.0f;
    float h_output_cpu = 0.0f;

    // 初始化输入数据（随机值 [0,1)）
    srand(42);  // 固定随机种子
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)rand() / RAND_MAX;
        h_output_cpu += h_A[i];
    }

    // 分配设备内存
    float *d_A, *d_output;
    cudaMalloc(&d_A, N * sizeof(float));
    cudaMalloc(&d_output, sizeof(float));

    // 初始化输出内存为 0（关键步骤！）
    cudaMemset(d_output, 0, sizeof(float));

    // 拷贝输入数据到设备
    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);

    // 执行归约
    reduce(d_A, d_output, N);

    // 拷贝结果回主机
    cudaMemcpy(&h_output_gpu, d_output, sizeof(float), cudaMemcpyDeviceToHost);

    // 打印结果
    printf("CPU Sum: %.6f\n", h_output_cpu);
    printf("GPU Sum: %.6f\n", h_output_gpu);

    const float eps = 1e-2;
    bool pass = fabs(h_output_gpu - h_output_cpu) < eps;
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 释放内存
    cudaFree(d_A);
    cudaFree(d_output);
}

int main() {
    test_reduce();
    return 0;
}

In [ ]:
!nvcc reduceBlock.cu -o reduceBlock -arch=sm_75 && ./reduceBlock

### Warp Shuffle 折半

warp 内 thread 是同步的 → 不需 `__syncthreads()`，且通过寄存器交换数据更快。
- `__shfl_down_sync(mask, val, offset)`：从 `lane_id + offset` 拷贝 `val`
- 流程：warp 内 reduce → warp_sum 写入 SMEM → warp0 做 block 级 reduce
- 设 `BLOCKSIZE = WARPSIZE * WARPSIZE`，第二次 SMEM 不超过 WARPSIZE

In [ ]:
%%writefile reduceWarp.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cstdlib>

#define BLOCKSIZE 1024
#define WARPSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void reduceWrapKernel(float *A, float *output, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int tid = threadIdx.x;
    int wrap_id = tid / WARPSIZE;
    int lane_id = tid % WARPSIZE;
    __shared__ float wrap_sum[BLOCKSIZE / WARPSIZE];

    // --- Step 1: Warp 内归约 ---
    float val = (idx < N) ? A[idx] : 0.0f;
    for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xFFFFFFFF, val, offset);
    }

    // --- Step 2: Warp 间归约 ---
    if (lane_id == 0) {
        wrap_sum[wrap_id] = val;
    }
    __syncthreads();

    // 第一个 Warp 负责 Block 内归约
    if (wrap_id == 0) {
        int num_wrap = BLOCKSIZE / WARPSIZE;
        float block_val = (lane_id < num_wrap) ? wrap_sum[lane_id] : 0.0f;
        for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
            block_val += __shfl_down_sync(0xFFFFFFFF, block_val, offset);
        }
        if (lane_id == 0) {
            atomicAdd(output, block_val);
        }
    }
}

void reduce(float *A, float *output, int N) {
    dim3 threadPerBlock(BLOCKSIZE);
    dim3 blockPerGrid(CEIL(N, BLOCKSIZE));
    reduceWrapKernel<<<blockPerGrid, threadPerBlock>>>(A, output, N);
    cudaDeviceSynchronize();
}

// 测试函数（验证 N=10000）
void test_reduce() {
    const int N = 10000;
    float h_A[N];
    float h_output_gpu = 0.0f;
    float h_output_cpu = 0.0f;

    // 初始化输入数据（随机值 [0,1)）
    srand(42);  // 固定随机种子
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)rand() / RAND_MAX;
        h_output_cpu += h_A[i];
    }

    // 分配设备内存
    float *d_A, *d_output;
    cudaMalloc(&d_A, N * sizeof(float));
    cudaMalloc(&d_output, sizeof(float));

    // 初始化输出内存为 0（关键步骤！）
    cudaMemset(d_output, 0, sizeof(float));

    // 拷贝输入数据到设备
    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);

    // 执行归约
    reduce(d_A, d_output, N);

    // 拷贝结果回主机
    cudaMemcpy(&h_output_gpu, d_output, sizeof(float), cudaMemcpyDeviceToHost);

    // 打印结果
    printf("CPU Sum: %.6f\n", h_output_cpu);
    printf("GPU Sum: %.6f\n", h_output_gpu);

    // 验证结果（调整容差为 1e-2）
    const float eps = 1e-2;
    bool pass = fabs(h_output_gpu - h_output_cpu) < eps;
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 释放内存
    cudaFree(d_A);
    cudaFree(d_output);
}

int main() {
    test_reduce();
    return 0;
}

In [ ]:
!nvcc reduceWarp.cu -o reduceWarp -arch=sm_75 && ./reduceWarp

### Warp Shuffle + Float4

在 warp 版基础上，每 thread 一次取 4 个 float：
- `BLOCKSIZE` 256（每 thread 处理 4 元素，总计算量同 1024）
- 多一步 `vec_val.x + .y + .z + .w` 的预加，再走 warp 流程

In [ ]:
%%writefile reduceWrapV4.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cstdlib>
#include <cassert>

#define BLOCKSIZE 256
#define WARPSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)
#define FLOAT4(value) (*reinterpret_cast<float4*>(&(value)))

__global__ void reduceWrapTiledV4Kernel(float *A, float *output, int N) {
    int tid = threadIdx.x;
    int wrap_id = tid / WARPSIZE;
    int lane_id = tid % WARPSIZE;
    int start_idx = (blockIdx.x * blockDim.x + threadIdx.x) * 4;
    __shared__ float wrap_sum[BLOCKSIZE / WARPSIZE];

    float4 vec_val = (start_idx + 3 < N) ? FLOAT4(A[start_idx]) : make_float4(0, 0, 0, 0);
    float val = vec_val.x + vec_val.y + vec_val.z + vec_val.w;

    for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xFFFFFFFF, val, offset);
    }

    if (lane_id == 0) {
        wrap_sum[wrap_id] = val;
    }
    __syncthreads();

    if (wrap_id == 0) {
        int num_wrap = BLOCKSIZE / WARPSIZE;
        float block_val = (lane_id < num_wrap) ? wrap_sum[lane_id] : 0;
        for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
            block_val += __shfl_down_sync(0xFFFFFFFF, block_val, offset);
        }
        if (lane_id == 0) {
            atomicAdd(output, block_val);
        }
    }
}

void reduce(float *A, float *output, int N) {
    dim3 threadPerBlock(BLOCKSIZE);
    dim3 blockPerGrid(CEIL(N, 4 * BLOCKSIZE));
    reduceWrapTiledV4Kernel<<<blockPerGrid, threadPerBlock>>>(A, output, N);
    cudaDeviceSynchronize();
}

// 测试函数（强制 N 为 4 的倍数）
void test_reduce() {
    const int N = 10000;  // 必须为 4 的倍数
    assert(N % 4 == 0 && "N must be a multiple of 4");

    float h_A[N];
    float h_output_gpu = 0.0f;
    float h_output_cpu = 0.0f;

    // 初始化输入数据（随机值 [0,1)）
    srand(42);
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)rand() / RAND_MAX;
        h_output_cpu += h_A[i];
    }

    // 分配设备内存
    float *d_A, *d_output;
    cudaMalloc(&d_A, N * sizeof(float));
    cudaMalloc(&d_output, sizeof(float));

    // 初始化输出内存为 0
    cudaMemset(d_output, 0, sizeof(float));

    // 拷贝输入数据到设备
    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);

    // 执行归约
    reduce(d_A, d_output, N);

    // 拷贝结果回主机
    cudaMemcpy(&h_output_gpu, d_output, sizeof(float), cudaMemcpyDeviceToHost);

    // 打印结果
    printf("CPU Sum: %.6f\n", h_output_cpu);
    printf("GPU Sum: %.6f\n", h_output_gpu);

    // 验证结果（允许更高容差）
    const float eps = 1e-4;
    bool pass = fabs(h_output_gpu - h_output_cpu) < eps;
    printf("\nTest: %s\n", pass ? "PASS" : "FAIL");

    // 释放内存
    cudaFree(d_A);
    cudaFree(d_output);
}

int main() {
    test_reduce();
    return 0;
}

!nvcc reduceWrapV4.cu -o reduceWrapV4 -arch=sm_75 && ./reduceWrapV4

In [ ]:
!nvcc reduceWrapV4.cu -o reduceWrapV4 -arch=sm_75 && ./reduceWrapV4

## 05 - Reduce 2D（行/列规约）

考察频率：中。

两个坑点：
- **dim/index 顺序**：`threadIdx.x` 对应 col，`threadIdx.y` 对应 row
- **Col reduce 不能 Float4**：列方向数据物理上不连续，无法合并访存
- **Col reduce 不要用 `(1, BLOCKSIZE)`**：warp 31/32 thread 浪费

### Col Reduce

- `blockPerGrid = (N, CEIL(M, BLOCKSIZE))` — 每个 block 处理 1 列的一段
- `threadPerBlock = (BLOCKSIZE,)` — 1D，避免 warp 浪费
- index：`col = bid.x`，`row = bid.y * bdim.x + tid.x`

In [ ]:
%%writefile reduce_cols.cu
#include <stdio.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <cmath>

#define BLOCKSIZE 1024
#define WARPSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void reduceColsKernel(const float* A, float* output, int M, int N) {
  int col = blockIdx.x;
  int global_row = blockIdx.y * blockDim.x + threadIdx.x;
  int tid = threadIdx.x;

  int wrap_id = tid/WARPSIZE;
  int lane_id = tid%WARPSIZE;
  __shared__ float wrap_sum[BLOCKSIZE/WARPSIZE];

  // in-wrap reduce
  float val = (global_row < M) ? A[global_row * N + col] : 0;
  for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
    val += __shfl_down_sync(0xFFFFFFFF, val, offset);
  }
  if (lane_id == 0) {
    wrap_sum[wrap_id] = val;
  }
  __syncthreads();

  // inter-wrap reduce
  if (wrap_id == 0) {
    float tmp_sum = (lane_id < (BLOCKSIZE/WARPSIZE)) ? wrap_sum[lane_id] : 0;
    for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
      tmp_sum += __shfl_down_sync(0xFFFFFFFF, tmp_sum, offset);
    }
    if (lane_id == 0) {
      atomicAdd(&output[col], tmp_sum);
    }
  }
}

void reduce(const float *A, float *B_col, float *B_row, int M, int N) {
  dim3 col_threadPerBlock(BLOCKSIZE);
  dim3 col_blockPerGrid(N, CEIL(M, BLOCKSIZE));
  reduceColsKernel<<<col_blockPerGrid, col_threadPerBlock>>>(A, B_col, M, N);
  cudaDeviceSynchronize();
}

// 辅助函数：打印矩阵
void print_matrix(const char* name, const float* data, int rows, int cols) {
  printf("%s (%dx%d):\n", name, rows, cols);
  for (int i = 0; i < rows; ++i) {
    for (int j = 0; j < cols; ++j) {
      printf("%6.1f ", data[i * cols + j]);
    }
    printf("\n");
  }
}

void test_case(int M, int N, const float* h_A) {
  // CPU计算
  float* h_B_col_cpu = new float[N]();
  for (int col = 0; col < N; ++col) {
    for (int row = 0; row < M; ++row) {
      h_B_col_cpu[col] += h_A[row * N + col];
    }
  }

  // GPU计算
  float *d_A, *d_B_col;
  cudaMalloc(&d_A, M*N*sizeof(float));
  cudaMalloc(&d_B_col, N*sizeof(float));
  cudaMemset(d_B_col, 0, N*sizeof(float));

  cudaMemcpy(d_A, h_A, M*N*sizeof(float), cudaMemcpyHostToDevice);
  reduce(d_A, d_B_col, nullptr, M, N);

  float* h_B_col_gpu = new float[N];
  cudaMemcpy(h_B_col_gpu, d_B_col, N*sizeof(float), cudaMemcpyDeviceToHost);

  // 打印结果
  print_matrix("Matrix A", h_A, M, N);
  printf("\nCPU Sums: ");
  for (int i = 0; i < N; ++i) printf("%.1f ", h_B_col_cpu[i]);
  printf("\nGPU Sums: ");
  for (int i = 0; i < N; ++i) printf("%.1f ", h_B_col_gpu[i]);

  // 验证结果
  bool pass = true;
  for (int i = 0; i < N; ++i) {
    if (fabs(h_B_col_cpu[i] - h_B_col_gpu[i]) > 1e-3) {
      pass = false;
      break;
    }
  }
  printf("\n\nTest: %s\n\n", pass ? "PASS" : "FAIL");

  // 释放资源
  delete[] h_B_col_cpu;
  delete[] h_B_col_gpu;
  cudaFree(d_A);
  cudaFree(d_B_col);
}

void test_reduce_cols() {
  // 测试用例1：小矩阵
  {
    const int M = 2, N = 4;
    float h_A[M*N] = {1,2,3,4,5,6,7,8};
    printf("==== Test Case 1 ====\n");
    test_case(M, N, h_A);
  }

  // 测试用例2：随机矩阵
  {
    const int M = 3, N = 2;
    float h_A[M*N];
    srand(42);
    for (int i = 0; i < M*N; ++i) {
      h_A[i] = (float)rand() / RAND_MAX * 10;
    }
    printf("==== Test Case 2 ====\n");
    test_case(M, N, h_A);
  }
}

int main() {
  test_reduce_cols();
  return 0;
}

In [ ]:
!nvcc reduce_cols.cu -o reduce_cols -arch=sm_75 && ./reduce_cols

### Row Reduce

- `blockPerGrid = (CEIL(N, BLOCKSIZE), M)`
- index：`row = bid.y`，`col = bid.x * bdim.x + tid.x`
- 行方向数据连续，可以加 Float4 进一步加速

In [ ]:
%%writefile reduce_rows.cu
#include <stdio.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <cmath>

#define BLOCKSIZE 1024
#define WARPSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void reduceRowsKernel(const float* A, float* output, int M, int N) {
  int row = blockIdx.y;
  int global_col = blockIdx.x * blockDim.x + threadIdx.x;
  int tid = threadIdx.x;

  int wrap_id = tid/WARPSIZE;
  int lane_id = tid%WARPSIZE;
  __shared__ float wrap_sum[BLOCKSIZE/WARPSIZE];

  // in-wrap reduce
  float val = (global_col < N) ? A[row * N + global_col] : 0.0f;
  for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
    val += __shfl_down_sync(0xFFFFFFFF, val, offset);
  }
  if (lane_id == 0) {
    wrap_sum[wrap_id] = val;
  }
  __syncthreads();

  // inter-wrap reduce
  if (wrap_id == 0) {
    float tmp_sum = (lane_id < (BLOCKSIZE/WARPSIZE)) ? wrap_sum[lane_id] : 0.0f;
    for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
      tmp_sum += __shfl_down_sync(0xFFFFFFFF, tmp_sum, offset);
    }
    if (lane_id == 0) {
      atomicAdd(&output[row], tmp_sum);
    }
  }
}

void reduce(const float *A, float *B_col, float *B_row, int M, int N) {
  dim3 row_threadPerBlock(BLOCKSIZE);
  dim3 row_blockPerGrid(CEIL(N, BLOCKSIZE), M);
  reduceRowsKernel<<<row_blockPerGrid, row_threadPerBlock>>>(A, B_row, M, N);
  cudaDeviceSynchronize();
}

void print_matrix(const char* name, const float* data, int rows, int cols) {
  printf("%s (%dx%d):\n", name, rows, cols);
  for (int i = 0; i < rows; ++i) {
    for (int j = 0; j < cols; ++j) {
      printf("%6.1f ", data[i * cols + j]);
    }
    printf("\n");
  }
}

void test_row_reduce() {
  // 测试用例1：小矩阵
  {
    const int M = 2, N = 4;
    float h_A[M][N] = {{1,2,3,4}, {5,6,7,8}};
    float h_B_row_cpu[M] = {10, 26};
    float *h_B_row_gpu = new float[M];

    float *d_A, *d_B_row;
    cudaMalloc(&d_A, M*N*sizeof(float));
    cudaMalloc(&d_B_row, M*sizeof(float));
    cudaMemset(d_B_row, 0, M*sizeof(float));
    cudaMemcpy(d_A, h_A, M*N*sizeof(float), cudaMemcpyHostToDevice);

    reduce(d_A, nullptr, d_B_row, M, N);
    cudaMemcpy(h_B_row_gpu, d_B_row, M*sizeof(float), cudaMemcpyDeviceToHost);

    print_matrix("Matrix A", (float*)h_A, M, N);
    printf("\nCPU Row Sums: ");
    for (int i = 0; i < M; ++i) printf("%.1f ", h_B_row_cpu[i]);
    printf("\nGPU Row Sums: ");
    for (int i = 0; i < M; ++i) printf("%.1f ", h_B_row_gpu[i]);

    bool pass = true;
    for (int i = 0; i < M; ++i) {
      if (fabs(h_B_row_gpu[i] - h_B_row_cpu[i]) > 1e-3) {
        pass = false;
        break;
      }
    }
    printf("\n\nTest: %s\n\n", pass ? "PASS" : "FAIL");

    delete[] h_B_row_gpu;
    cudaFree(d_A);
    cudaFree(d_B_row);
  }

  // 测试用例2：随机大矩阵（修复版本）
  {
    const int M = 1000, N = 2048;
    float *h_A = new float[M*N];
    float *h_B_row_cpu = new float[M]();
    float *h_B_row_gpu = new float[M](); // 初始化GPU结果为0

    // 生成带可控和的数据
    srand(42);
    for (int i = 0; i < M; ++i) {
      float row_sum = 0.0f;
      for (int j = 0; j < N; ++j) {
        h_A[i*N + j] = (j % 2) ? 0.5f : 1.5f; // 交替值确保精度
        row_sum += h_A[i*N + j];
      }
      h_B_row_cpu[i] = row_sum;
    }

    float *d_A, *d_B_row;
    cudaMalloc(&d_A, M*N*sizeof(float));
    cudaMalloc(&d_B_row, M*sizeof(float));
    cudaMemset(d_B_row, 0, M*sizeof(float));
    cudaMemcpy(d_A, h_A, M*N*sizeof(float), cudaMemcpyHostToDevice);

    reduce(d_A, nullptr, d_B_row, M, N);
    cudaMemcpy(h_B_row_gpu, d_B_row, M*sizeof(float), cudaMemcpyDeviceToHost);

    // 计算最大相对误差
    float max_rel_err = 0.0f;
    for (int i = 0; i < M; ++i) {
      float err = fabs(h_B_row_gpu[i] - h_B_row_cpu[i]) / h_B_row_cpu[i];
      if (err > max_rel_err) max_rel_err = err;
    }

    printf("==== Large Matrix Test ====");
    printf("\nMax Relative Error: %.6f%%", max_rel_err * 100);
    printf("\nTest: %s\n", (max_rel_err < 1e-4) ? "PASS" : "FAIL");

    delete[] h_A;
    delete[] h_B_row_cpu;
    delete[] h_B_row_gpu;
    cudaFree(d_A);
    cudaFree(d_B_row);
  }
}

int main() {
  test_row_reduce();
  return 0;
}

In [ ]:
!nvcc reduce_rows.cu -o reduce_rows -arch=sm_75 && ./reduce_rows

## 06 - Softmax

考察频率：低。

3 阶段（避免溢出，先减最大值）：
1. **求行 max** — warp reduce
2. **求 $\sum \exp(x - max)$** — warp reduce
3. **elementwise**：$y_i = \exp(x_i - max) / \text{sum}$

每行开一个 block，用 1 个 warp 做整行的 reduce（大矩阵时改成两阶段 reduce）。

### 2D Softmax

3 阶段都用 `WARPSIZE` 步长循环：每个 lane 跨 `WARPSIZE` 步处理多个元素。

In [ ]:
%%writefile softmax2d.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>
#include <float.h>
#include <vector>

#define WARPSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void softmax2DKernel(float* A, float* B, int M, int N) {
  __shared__ float s_max_val;
  __shared__ float s_sum_val;

  int row = blockIdx.x;
  int lane_id = threadIdx.x % WARPSIZE;
  int iteration = CEIL(N, WARPSIZE);

  // --- stage 1: 求最大值 ---
  float max_val = -FLT_MAX;
  for (int i = 0; i < iteration; i++){
    int col = i * WARPSIZE + lane_id;
    if (col < N) max_val = fmaxf(max_val, A[row * N + col]);
  }
  // wrap内求max
  for(int offset = WARPSIZE/2; offset > 0; offset >>= 1) {
  	max_val = fmaxf(max_val, __shfl_down_sync(0xFFFFFFFF, max_val, offset));
  }
  if (lane_id == 0) s_max_val = max_val;
  __syncthreads();

  // --- stage 2: 求exp指数和 ---
  float exp_sum = 0;
  for (int i = 0; i < iteration; i++){
    int col = i * WARPSIZE + lane_id;
    if (col < N) exp_sum += expf(A[row * N + col] - s_max_val);
  }
  // wrap内求指数sum
  for(int offset = WARPSIZE/2; offset > 0; offset >>= 1) {
    exp_sum += __shfl_down_sync(0xFFFFFFFF, exp_sum, offset);
  }
  // 写入SMEM
  if(lane_id == 0) s_sum_val = exp_sum;
  __syncthreads();

  // --- stage 3: 计算Softmax ---
  for(int i = 0; i < iteration; i++){
    int col = i * WARPSIZE + lane_id;
    if(col < N) B[row * N + col] = expf(A[row * N + col] - s_max_val) / s_sum_val;
  }
}

void softmax2D(float* A, float* B, int M, int N) {
    dim3 threadPerBlock(WARPSIZE);
    dim3 blockPerGrid(M);
    softmax2DKernel<<<blockPerGrid, threadPerBlock>>>(A, B, M, N);
    cudaDeviceSynchronize();
}

// 打印前4x4矩阵内容
void print_matrix_head(const char* name, const float* data, int rows, int cols) {
  printf("%s (first 4x4):\n", name);
  int print_rows = rows < 4 ? rows : 4;
  int print_cols = cols < 4 ? cols : 4;
  for (int i = 0; i < print_rows; ++i) {
    for (int j = 0; j < print_cols; ++j) {
      printf("%8.4f ", data[i * cols + j]);
    }
    printf("\n");
  }
}

// CPU参考实现
void cpu_softmax(const float* input, float* output, int M, int N) {
  for (int i = 0; i < M; ++i) {
    float max_val = -FLT_MAX;
    for (int j = 0; j < N; ++j)
      max_val = fmaxf(max_val, input[i*N + j]);

    float exp_sum = 0;
    for (int j = 0; j < N; ++j)
      exp_sum += expf(input[i*N + j] - max_val);

    for (int j = 0; j < N; ++j)
      output[i*N + j] = expf(input[i*N + j] - max_val) / exp_sum;
  }
}

void test_softmax_case(const float* h_input, int M, int N, float tolerance) {
  std::vector<float> h_output_cpu(M * N);
  std::vector<float> h_output_gpu(M * N);
  float *d_input, *d_output;

  // CPU计算
  cpu_softmax(h_input, h_output_cpu.data(), M, N);

  // GPU计算
  cudaMalloc(&d_input, M*N*sizeof(float));
  cudaMalloc(&d_output, M*N*sizeof(float));
  cudaMemcpy(d_input, h_input, M*N*sizeof(float), cudaMemcpyHostToDevice);

  softmax2D(d_input, d_output, M, N);

  cudaMemcpy(h_output_gpu.data(), d_output, M*N*sizeof(float), cudaMemcpyDeviceToHost);

  // 打印样本数据
  print_matrix_head("Input", h_input, M, N);
  print_matrix_head("CPU Result", h_output_cpu.data(), M, N);
  print_matrix_head("GPU Result", h_output_gpu.data(), M, N);

  // 验证结果
  bool pass = true;
  for (int i = 0; i < M*N; ++i) {
    float err = fabs(h_output_cpu[i] - h_output_gpu[i]);
    if (err > tolerance) {
      printf("Mismatch at (%d,%d): CPU=%.6f GPU=%.6f (err=%.6f)\n",
             i/N, i%N, h_output_cpu[i], h_output_gpu[i], err);
      pass = false;
      break;
    }
  }
  printf("Test: %s\n\n", pass ? "PASS" : "FAIL");

  cudaFree(d_input);
  cudaFree(d_output);
}

int main() {
  // 测试用例1：小矩阵测试
  {
    const int M = 2, N = 4;
    float h_input[M][N] = {
      {1.0f, 2.0f, 3.0f, 4.0f},
      {5.0f, 6.0f, 7.0f, 8.0f}
    };
    printf("==== Test Case 1: Small Matrix ====\n");
    test_softmax_case((float*)h_input, M, N, 1e-5f);
  }

  // 测试用例2：数值稳定性测试
  {
    const int M = 3, N = 2;
    std::vector<float> h_input(M*N);
    srand(42);
    for (int i = 0; i < M*N; ++i)
      h_input[i] = (float)(rand() % 10000 - 5000); // [-5000, 5000]
    printf("==== Test Case 2: Numerical Stability ====\n");
    test_softmax_case(h_input.data(), M, N, 1e-4f);
  }

  // 测试用例3：大矩阵测试（自动处理M限制）
  {
    const int M = 50000, N = 256; // M < 65535
    std::vector<float> h_input(M*N);
    srand(42);
    for (int i = 0; i < M*N; ++i)
      h_input[i] = (float)rand()/RAND_MAX * 20.0f - 10.0f; // [-10,10]
    printf("==== Test Case 3: Large Matrix ====\n");
    test_softmax_case(h_input.data(), M, N, 1e-4f);
  }

  return 0;
}

In [ ]:
!nvcc softmax2d.cu -o softmax2d -arch=sm_75 && ./softmax2d

### 2D Softmax + Float4

3 次 reduce/elementwise 都 vec4 化。要求列数是 4 的倍数。

> 实际面试不太建议手写这个 — 3 次 vec4 太烦了。

In [ ]:
%%writefile softmax2dV4.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>
#include <float.h>
#include <vector>

#define WARPSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)
#define FLOAT4(ptr) (*(reinterpret_cast<float4*>(&(ptr))))

__global__ void softmax2DV4Kernel(float* A, float* B, int M, int N){
  __shared__ float s_max_val;
  __shared__ float s_sum_val;

  int row = blockIdx.x;
  int lane_id = threadIdx.x % WARPSIZE;
  int iteration = CEIL(N, WARPSIZE*4);

  // --- stage 1: 向量化求最大值 ---
  float max_val = -FLT_MAX;
  for (int i = 0; i < iteration; i++){
    int col = (i * WARPSIZE + lane_id)*4;
    if (col + 3 < N){
      float4 vec = FLOAT4(A[row * N + col]);
      max_val = fmaxf(max_val, fmaxf(fmaxf(vec.x, vec.y), fmaxf(vec.z, vec.w)));
    }
  }
  // wrap内求max
  for(int offset = WARPSIZE/2; offset > 0; offset >>= 1) {
  	max_val = fmaxf(max_val, __shfl_down_sync(0xFFFFFFFF, max_val, offset));
  }
  // 写入SMEM
  if (lane_id == 0) s_max_val = max_val;
  __syncthreads();

  // --- stage 2: 向量化求exp指数和 ---
  float exp_sum = 0;
  for (int i = 0; i < iteration; i++) {
    int col = (i * WARPSIZE + lane_id) * 4;
    if (col + 3 < N) {
      float4 vec = FLOAT4(A[row * N + col]);
      exp_sum += expf(vec.x - s_max_val) + expf(vec.y - s_max_val)
               + expf(vec.z - s_max_val) + expf(vec.w - s_max_val);
    }
  }
  // wrap内求指数sum
  for(int offset = WARPSIZE/2; offset > 0; offset >>= 1) {
    exp_sum += __shfl_down_sync(0xFFFFFFFF, exp_sum, offset);
  }
  // 写入SMEM
  if(lane_id == 0) s_sum_val = exp_sum;
  __syncthreads();

  // --- stage 3: 向量化计算Softmax ---
  const float sum_inv = 1.0f / s_sum_val;
  for (int i = 0; i < iteration; i++) {
    int col = (i * WARPSIZE + lane_id) * 4;
    if (col + 3 < N) {
      float4 vec = FLOAT4(A[row * N + col]);
      float4 result;
      result.x = expf(vec.x - s_max_val) * sum_inv;
      result.y = expf(vec.y - s_max_val) * sum_inv;
      result.z = expf(vec.z - s_max_val) * sum_inv;
      result.w = expf(vec.w - s_max_val) * sum_inv;
      FLOAT4(B[row * N + col]) = result;
    }
  }
}

void softmax2D(float* A, float* B, int M, int N) {
    dim3 threadPerBlock(WARPSIZE);
    dim3 blockPerGrid(M);
    softmax2DV4Kernel<<<blockPerGrid, threadPerBlock>>>(A, B, M, N);
    cudaDeviceSynchronize();
}

// 打印前4行前4列数据
void print_matrix_head(const char* name, const float* data, int M, int N) {
  printf("%s (first 4x4):\n", name);
  for (int i = 0; i < std::min(4, M); ++i) {
    for (int j = 0; j < std::min(4, N); ++j) {
      printf("%8.4f ", data[i*N + j]);
    }
    printf("\n");
  }
}

// CPU参考实现（处理任意N值）
void cpu_softmax_v4(const float* input, float* output, int M, int N) {
  for (int i = 0; i < M; ++i) {
    float max_val = -FLT_MAX;
    for (int j = 0; j < N; ++j)
      max_val = fmaxf(max_val, input[i*N + j]);

    float exp_sum = 0;
    for (int j = 0; j < N; ++j)
      exp_sum += expf(input[i*N + j] - max_val);

    for (int j = 0; j < N; ++j)
      output[i*N + j] = expf(input[i*N + j] - max_val) / exp_sum;
  }
}

void test_softmax_v4_case(int M, int N) {
  // 确保N是4的倍数
  N = (N + 3) / 4 * 4;

  std::vector<float> h_input(M*N);
  std::vector<float> h_output_cpu(M*N);
  std::vector<float> h_output_gpu(M*N);

  // 生成测试数据（数值范围[-10, 10]）
  srand(42);
  for (int i = 0; i < M*N; ++i)
    h_input[i] = (float)(rand() % 2000) / 100.0f - 10.0f;

  // CPU计算
  cpu_softmax_v4(h_input.data(), h_output_cpu.data(), M, N);

  // GPU计算
  float *d_input, *d_output;
  cudaMalloc(&d_input, M*N*sizeof(float));
  cudaMalloc(&d_output, M*N*sizeof(float));
  cudaMemcpy(d_input, h_input.data(), M*N*sizeof(float), cudaMemcpyHostToDevice);

  softmax2D(d_input, d_output, M, N);

  cudaMemcpy(h_output_gpu.data(), d_output, M*N*sizeof(float), cudaMemcpyDeviceToHost);

  // 打印样本数据
  print_matrix_head("Input", h_input.data(), M, N);
  print_matrix_head("CPU Result", h_output_cpu.data(), M, N);
  print_matrix_head("GPU Result", h_output_gpu.data(), M, N);

  // 验证结果（放宽容差到1e-5）
  bool pass = true;
  for (int i = 0; i < M*N; ++i) {
    float err = fabs(h_output_cpu[i] - h_output_gpu[i]);
    if (err > 1e-5) {
      printf("Mismatch at (%d,%d): CPU=%.6f GPU=%.6f (err=%.6f)\n",
             i/N, i%N, h_output_cpu[i], h_output_gpu[i], err);
      pass = false;
      break;
    }
  }
  printf("Test: %s\n\n", pass ? "PASS" : "FAIL");

  cudaFree(d_input);
  cudaFree(d_output);
}

int main() {
  // 测试用例1：小矩阵（N=4）
  {
    const int M = 2, N = 4;
    printf("==== Test Case 1: Small Matrix (4x4) ====\n");
    test_softmax_v4_case(M, N);
  }

  // 测试用例2：中规模矩阵（自动对齐到4的倍数）
  {
    const int M = 8, N = 10; // 实际N=12
    printf("\n==== Test Case 2: Medium Matrix (auto align) ====\n");
    test_softmax_v4_case(M, N);
  }

  // 测试用例3：大规模矩阵（M=50000, N=256）
  {
    const int M = 50000, N = 256;
    printf("\n==== Test Case 3: Large Matrix ====\n");
    test_softmax_v4_case(M, N);
  }

  return 0;
}

In [ ]:
!nvcc softmax2dV4.cu -o softmax2dV4 -arch=sm_75 && ./softmax2dV4

## 07 - Conv

考察频率：低。忽略 stride，output size = `N - K + 1`。

- **Naive**：每 thread 算 1 个 output，循环 K（或 K×K）次乘加
- **Tiled**：每 block 加载 BLOCKSIZE 个输入到 SMEM，但只有前 `BLOCKSIZE - K + 1` 个 thread 写 output
- 用 `extern __shared__ float s[];` 动态分配 SMEM 大小

### 1D Conv - Naive

In [ ]:
%%writefile conv1d.cu
#include <stdio.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <cmath>

#define BLOCKSIZE 1024
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void conv1DKernel(const float *input, const float *kernel, float *output, int N, int K) {
  int tid = blockIdx.x * blockDim.x + threadIdx.x;
  if (tid >= N - K + 1) return;

  float tmp = 0;
  for (int i = 0; i < K; i++) {
    tmp += input[tid+i] * kernel[i];
  }
  output[tid] = tmp;
}

void conv1D(const float *input, const float *kernel, float *output, int N, int K) {
  int output_size = N - K + 1;
  if (output_size <= 0) return; // 添加边界保护

  dim3 threadPerBlock(BLOCKSIZE);
  dim3 blockPerGrid(CEIL(output_size, BLOCKSIZE));
  conv1DKernel<<<blockPerGrid, threadPerBlock>>>(input, kernel, output, N, K);
  cudaDeviceSynchronize();
}

void print_vector(const char* name, const float* data, int size) {
  printf("%s (%d):\n", name, size);
  for (int i = 0; i < size; ++i) {
    printf("%6.2f ", data[i]);
    if ((i+1) % 16 == 0) printf("\n");
  }
  printf("\n");
}

void cpu_conv1d(const float* input, const float* kernel, float* output, int N, int K) {
  int output_size = N - K + 1;
  for (int i = 0; i < output_size; ++i) {
    float sum = 0;
    for (int j = 0; j < K; ++j) {
      sum += input[i+j] * kernel[j];
    }
    output[i] = sum;
  }
}

void test_conv_case(const float* h_input, const float* h_kernel, int N, int K) {
  const int output_size = N - K + 1;
  if (output_size <= 0) {
    printf("Invalid case: N=%d, K=%d\n", N, K);
    return;
  }

  // 分配内存
  float *h_output_cpu = new float[output_size];
  float *h_output_gpu = new float[output_size];
  float *d_input, *d_kernel, *d_output;

  // CPU计算
  cpu_conv1d(h_input, h_kernel, h_output_cpu, N, K);

  // GPU计算
  cudaMalloc(&d_input, N*sizeof(float));
  cudaMalloc(&d_kernel, K*sizeof(float));
  cudaMalloc(&d_output, output_size*sizeof(float));

  cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_kernel, h_kernel, K*sizeof(float), cudaMemcpyHostToDevice);

  conv1D(d_input, d_kernel, d_output, N, K);

  cudaMemcpy(h_output_gpu, d_output, output_size*sizeof(float), cudaMemcpyDeviceToHost);

  // 打印结果
  print_vector("Input", h_input, N);
  print_vector("Kernel", h_kernel, K);
  print_vector("CPU Output", h_output_cpu, output_size);
  print_vector("GPU Output", h_output_gpu, output_size);

  // 验证结果
  bool pass = true;
  for (int i = 0; i < output_size; ++i) {
    if (fabs(h_output_cpu[i] - h_output_gpu[i]) > 1e-4) {
      printf("Mismatch at %d: CPU=%.4f, GPU=%.4f\n",
             i, h_output_cpu[i], h_output_gpu[i]);
      pass = false;
    }
  }
  printf("Test: %s\n\n", pass ? "PASS" : "FAIL");

  // 清理
  delete[] h_output_cpu;
  delete[] h_output_gpu;
  cudaFree(d_input);
  cudaFree(d_kernel);
  cudaFree(d_output);
}

int main() {
  // 测试用例1：小规模测试
  {
    const int N = 5, K = 3;
    float h_input[N] = {1, 2, 3, 4, 5};
    float h_kernel[K] = {0.5, 1.0, 0.5};
    printf("==== Test Case 1: Small Conv ====\n");
    test_conv_case(h_input, h_kernel, N, K);
  }

  // 测试用例2：边界测试
  {
    const int N = 1000, K = 200;
    float *h_input = new float[N];
    float *h_kernel = new float[K];

    // 生成随机数据
    srand(42);
    for (int i = 0; i < N; ++i) h_input[i] = (float)rand()/RAND_MAX;
    for (int i = 0; i < K; ++i) h_kernel[i] = (float)rand()/RAND_MAX;

    printf("\n==== Test Case 2: Large Conv ====\n");
    test_conv_case(h_input, h_kernel, N, K);

    delete[] h_input;
    delete[] h_kernel;
  }

  // 测试用例3：无效情况测试
  {
    printf("\n==== Test Case 3: Invalid Case ====\n");
    float dummy[3] = {0};
    test_conv_case(dummy, dummy, 2, 3); // N=2, K=3 => output_size=0
  }

  return 0;
}

In [ ]:
!nvcc conv1d.cu -o conv1d -arch=sm_75 && ./conv1d

### 1D Conv - Tiled

In [ ]:
%%writefile conv1d_tiled.cu
#include <stdio.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <cmath>

#define BLOCKSIZE 1024
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void conv1DKernelTiled(const float *input, const float *kernel, float *output, int N, int K) {
  int O_Total = BLOCKSIZE - K + 1;
  extern __shared__ float s_input[];
  int tid = threadIdx.x;
  int block_start = blockIdx.x * O_Total;

  // 加载数据到共享内存
  s_input[tid] = (block_start + tid < N) ? input[block_start + tid] : 0.0f;
  __syncthreads();

  // 计算卷积
  int o_idx = block_start + tid;
  if (tid < O_Total) {
    float sum = 0.0f;
    #pragma unroll
    for (int i = 0; i < K; i++) {
      sum += s_input[tid + i] * kernel[i];
    }
    if (o_idx < N - K + 1) output[o_idx] = sum;
  }
}

void conv1D(const float *input, const float *kernel, float *output, int N, int K) {
  int output_size = N - K + 1;
  if (output_size <= 0) return;

  const int O_Total = BLOCKSIZE - K + 1;
  dim3 threadPerBlock(BLOCKSIZE);
  dim3 blockPerGrid(CEIL(output_size, O_Total));
  size_t shared_mem_size = BLOCKSIZE * sizeof(float);
  conv1DKernelTiled<<<blockPerGrid, threadPerBlock, shared_mem_size>>>(input, kernel, output, N, K);
  cudaDeviceSynchronize();
}

void print_vector(const char* name, const float* data, int size) {
  printf("%s (%d): [ ", name, size);
  for (int i = 0; i < std::min(10, size); ++i) {
    printf("%.2f ", data[i]);
  }
  (size > 10) ? printf("... ]\n") : printf("]\n");
}

void cpu_conv1d(const float* input, const float* kernel, float* output, int N, int K) {
  int output_size = N - K + 1;
  for (int i = 0; i < output_size; ++i) {
    float sum = 0.0f;
    for (int j = 0; j < K; ++j) {
      sum += input[i + j] * kernel[j];
    }
    output[i] = sum;
  }
}

void test_conv_case(const float* h_input, const float* h_kernel, int N, int K) {
  const int output_size = N - K + 1;
  if (output_size <= 0) {
    printf("Invalid case: N=%d, K=%d\n", N, K);
    return;
  }

  // 分配内存
  float *h_output_cpu = new float[output_size];
  float *h_output_gpu = new float[output_size];
  float *d_input, *d_kernel, *d_output;

  // CPU计算
  cpu_conv1d(h_input, h_kernel, h_output_cpu, N, K);

  // GPU计算
  cudaMalloc(&d_input, N*sizeof(float));
  cudaMalloc(&d_kernel, K*sizeof(float));
  cudaMalloc(&d_output, output_size*sizeof(float));

  cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_kernel, h_kernel, K*sizeof(float), cudaMemcpyHostToDevice);

  conv1D(d_input, d_kernel, d_output, N, K);

  cudaMemcpy(h_output_gpu, d_output, output_size*sizeof(float), cudaMemcpyDeviceToHost);

  // 打印结果
  print_vector("Input", h_input, N);
  print_vector("Kernel", h_kernel, K);
  print_vector("CPU Output", h_output_cpu, output_size);
  print_vector("GPU Output", h_output_gpu, output_size);

  // 验证结果
  bool pass = true;
  for (int i = 0; i < output_size; ++i) {
    if (fabs(h_output_cpu[i] - h_output_gpu[i]) > 1e-4) {
      printf("Mismatch at %d: CPU=%.4f, GPU=%.4f\n", i, h_output_cpu[i], h_output_gpu[i]);
      pass = false;
      break;
    }
  }
  printf("Test: %s\n\n", pass ? "PASS" : "FAIL");

  // 清理
  delete[] h_output_cpu;
  delete[] h_output_gpu;
  cudaFree(d_input);
  cudaFree(d_kernel);
  cudaFree(d_output);
}

int main() {
  // 测试用例1：精确匹配测试
  {
    const int N = 15, K = 3;
    float h_input[N] = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15};
    float h_kernel[K] = {0.5, 1.0, 0.5};
    printf("==== Test Case 1: Basic Test ====\n");
    test_conv_case(h_input, h_kernel, N, K);
  }

  // 测试用例2：边界块测试
  {
    const int N = BLOCKSIZE * 2 + 5, K = 5; // 特殊块处理
    float *h_input = new float[N];
    float *h_kernel = new float[K];

    // 生成序列数据
    for (int i = 0; i < N; ++i) h_input[i] = i + 1;
    for (int i = 0; i < K; ++i) h_kernel[i] = 1.0f;

    printf("\n==== Test Case 2: Block Boundary Test ====\n");
    test_conv_case(h_input, h_kernel, N, K);

    delete[] h_input;
    delete[] h_kernel;
  }

  // 测试用例3：随机大数据测试
  {
    const int N = 10000, K = 50;
    float *h_input = new float[N];
    float *h_kernel = new float[K];

    srand(42);
    for (int i = 0; i < N; ++i) h_input[i] = (float)rand()/RAND_MAX;
    for (int i = 0; i < K; ++i) h_kernel[i] = (float)rand()/RAND_MAX;

    printf("\n==== Test Case 3: Large Data Test ====\n");
    test_conv_case(h_input, h_kernel, N, K);

    delete[] h_input;
    delete[] h_kernel;
  }

  return 0;
}

In [ ]:
!nvcc conv1d_tiled.cu -o conv1d_tiled -arch=sm_75 && ./conv1d_tiled

### 2D Conv - Naive

In [ ]:
%%writefile conv2d_basic.cu
#include <stdio.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <cmath>

#define BLOCKSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void conv2DKernel(const float *input, const float *kernel, float *output,
                           int M, int N, int K) {
  // 原始kernel代码保持不变
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  if (row >= M - K + 1 || col >= N - K + 1) return;

  float tmp = 0;
  for (int i = 0; i < K; i++){
    for (int j = 0; j < K; j++){
      tmp += input[(row+i)*N + (col+j)] * kernel[i*K + j];
    }
  }
  output[row * (N - K + 1) + col] = tmp;
}

void conv2D(const float *input, const float *kernel, float *output,
          int M, int N, int K) {
  const int output_M = M - K + 1;
  const int output_N = N - K + 1;
  if (output_M <= 0 || output_N <= 0) return;

  dim3 threads(BLOCKSIZE, BLOCKSIZE);
  dim3 blocks(CEIL(output_N, threads.x), CEIL(output_M, threads.y));

  conv2DKernel<<<blocks, threads>>>(input, kernel, output, M, N, K);
  cudaDeviceSynchronize();
}

// ---------- 测试工具函数 ----------
void cpu_conv2d(const float* input, const float* kernel,
              float* output, int M, int N, int K) {
  const int out_M = M - K + 1;
  const int out_N = N - K + 1;

  for (int i = 0; i < out_M; ++i) {
    for (int j = 0; j < out_N; ++j) {
      float sum = 0.0f;
      for (int ki = 0; ki < K; ++ki) {
        for (int kj = 0; kj < K; ++kj) {
          sum += input[(i+ki)*N + (j+kj)] * kernel[ki*K + kj];
        }
      }
      output[i*out_N + j] = sum;
    }
  }
}

void print_matrix(const char* name, const float* data, int rows, int cols) {
  printf("%s (%dx%d):\n", name, rows, cols);
  for (int i = 0; i < std::min(3, rows); ++i) {
    for (int j = 0; j < std::min(3, cols); ++j) {
      printf("%6.2f ", data[i*cols + j]);
    }
    printf("\n");
  }
}

void test_case(const float* h_input, const float* h_kernel,
             int M, int N, int K) {
  const int out_M = M - K + 1;
  const int out_N = N - K + 1;

  if (out_M <= 0 || out_N <= 0) {
    printf("Invalid case: M=%d, N=%d, K=%d\n", M, N, K);
    return;
  }

  // 分配内存
  float *h_output_cpu = new float[out_M * out_N];
  float *h_output_gpu = new float[out_M * out_N];
  float *d_input, *d_kernel, *d_output;

  // CPU计算
  cpu_conv2d(h_input, h_kernel, h_output_cpu, M, N, K);

  // GPU计算
  cudaMalloc(&d_input, M*N*sizeof(float));
  cudaMalloc(&d_kernel, K*K*sizeof(float));
  cudaMalloc(&d_output, out_M*out_N*sizeof(float));

  cudaMemcpy(d_input, h_input, M*N*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_kernel, h_kernel, K*K*sizeof(float), cudaMemcpyHostToDevice);

  conv2D(d_input, d_kernel, d_output, M, N, K);

  cudaMemcpy(h_output_gpu, d_output, out_M*out_N*sizeof(float), cudaMemcpyDeviceToHost);

  // 打印结果
  print_matrix("Input", h_input, M, N);
  print_matrix("Kernel", h_kernel, K, K);
  print_matrix("CPU Output", h_output_cpu, out_M, out_N);
  print_matrix("GPU Output", h_output_gpu, out_M, out_N);

  // 验证结果
  bool pass = true;
  for (int i = 0; i < out_M*out_N; ++i) {
    if (fabs(h_output_cpu[i] - h_output_gpu[i]) > 1e-4) {
      printf("Mismatch at (%d,%d): CPU=%.4f GPU=%.4f\n",
            i/out_N, i%out_N, h_output_cpu[i], h_output_gpu[i]);
      pass = false;
      break;
    }
  }
  printf("Test: %s\n\n", pass ? "PASS" : "FAIL");

  // 清理资源
  delete[] h_output_cpu;
  delete[] h_output_gpu;
  cudaFree(d_input);
  cudaFree(d_kernel);
  cudaFree(d_output);
}

int main() {
  // 测试用例1：基本功能测试
  {
    const int M = 5, N = 5, K = 3;
    float input[] = {
      1,2,3,4,5,
      6,7,8,9,10,
      11,12,13,14,15,
      16,17,18,19,20,
      21,22,23,24,25
    };
    float kernel[] = { // 边缘检测核
      1, 0, -1,
      2, 0, -2,
      1, 0, -1
    };
    printf("==== Test Case 1: Basic Function ====\n");
    test_case(input, kernel, M, N, K);
  }

  // 测试用例2：边界条件测试
  {
    const int M = 33, N = 33, K = 5; // 测试块边界
    float *input = new float[M*N];
    float *kernel = new float[K*K];

    // 生成测试数据
    for (int i = 0; i < M*N; ++i) input[i] = i % 10;
    for (int i = 0; i < K*K; ++i) kernel[i] = 1.0f/(K*K);

    printf("\n==== Test Case 2: Block Boundary ====\n");
    test_case(input, kernel, M, N, K);

    delete[] input;
    delete[] kernel;
  }

  return 0;
}

In [ ]:
!nvcc conv2d_basic.cu -o conv2d_test -arch=sm_75 && ./conv2d_test

### 2D Conv - Tiled

In [ ]:
%%writefile conv2d_tiled.cu
#include <stdio.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <cmath>

#define BLOCKSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void conv2DKernelTiled(const float *input, const float *kernel, float *output,
                             int M, int N, int K) {
  // 保持用户原始代码不变（除共享内存访问方式）
  extern __shared__ float s_input[];
  const int O_Total = BLOCKSIZE - K + 1;

  const int x_start = blockIdx.y * O_Total;
  const int y_start = blockIdx.x * O_Total;
  const int x_tid = threadIdx.y;
  const int y_tid = threadIdx.x;

  // 将二维共享内存转换为一维访问
  const int s_idx = x_tid * BLOCKSIZE + y_tid;
  const int load_x = x_start + x_tid;
  const int load_y = y_start + y_tid;

  s_input[s_idx] = (load_x < M && load_y < N) ? input[load_x*N + load_y] : 0.0f;
  __syncthreads();

  if (x_tid < O_Total && y_tid < O_Total) {
    float sum = 0.0f;
    for (int i = 0; i < K; ++i) {
      for (int j = 0; j < K; ++j) {
        const int s_row = x_tid + i;
        const int s_col = y_tid + j;
        sum += s_input[s_row * BLOCKSIZE + s_col] * kernel[i*K + j];
      }
    }
    const int o_x = x_start + x_tid;
    const int o_y = y_start + y_tid;
    if (o_x < M-K+1 && o_y < N-K+1) {
      output[o_x*(N-K+1) + o_y] = sum;
    }
  }
}

void conv2D(const float *input, const float *kernel, float *output,
          int M, int N, int K) {
  if (M < K || N < K) return;

  const int output_M = M - K + 1;
  const int output_N = N - K + 1;
  const int O_Total = BLOCKSIZE - K + 1;

  dim3 threads(BLOCKSIZE, BLOCKSIZE);
  dim3 blocks(CEIL(output_N, O_Total), CEIL(output_M, O_Total));
  size_t shared_size = BLOCKSIZE * BLOCKSIZE * sizeof(float);

  conv2DKernelTiled<<<blocks, threads, shared_size>>>(input, kernel, output, M, N, K);
  cudaDeviceSynchronize();
}

// CPU参考实现
void cpu_conv2d(const float* input, const float* kernel,
               float* output, int M, int N, int K) {
  const int out_M = M - K + 1;
  const int out_N = N - K + 1;

  for (int i = 0; i < out_M; ++i) {
    for (int j = 0; j < out_N; ++j) {
      float sum = 0.0f;
      for (int ki = 0; ki < K; ++ki) {
        for (int kj = 0; kj < K; ++kj) {
          sum += input[(i+ki)*N + (j+kj)] * kernel[ki*K + kj];
        }
      }
      output[i*out_N + j] = sum;
    }
  }
}

void print_matrix(const char* name, const float* data, int rows, int cols) {
  printf("%s (%dx%d):\n", name, rows, cols);
  for (int i = 0; i < std::min(3, rows); ++i) {
    for (int j = 0; j < std::min(3, cols); ++j) {
      printf("%6.2f ", data[i*cols + j]);
    }
    printf("\n");
  }
}

void test_case(const float* h_input, const float* h_kernel,
              int M, int N, int K) {
  const int out_M = M - K + 1;
  const int out_N = N - K + 1;

  if (out_M <= 0 || out_N <= 0) {
    printf("Invalid case: M=%d, N=%d, K=%d\n", M, N, K);
    return;
  }

  // 分配内存
  float *h_output_cpu = new float[out_M * out_N];
  float *h_output_gpu = new float[out_M * out_N];
  float *d_input, *d_kernel, *d_output;

  // CPU计算
  cpu_conv2d(h_input, h_kernel, h_output_cpu, M, N, K);

  // GPU计算
  cudaMalloc(&d_input, M*N*sizeof(float));
  cudaMalloc(&d_kernel, K*K*sizeof(float));
  cudaMalloc(&d_output, out_M*out_N*sizeof(float));

  cudaMemcpy(d_input, h_input, M*N*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_kernel, h_kernel, K*K*sizeof(float), cudaMemcpyHostToDevice);

  conv2D(d_input, d_kernel, d_output, M, N, K);

  cudaMemcpy(h_output_gpu, d_output, out_M*out_N*sizeof(float), cudaMemcpyDeviceToHost);

  // 打印结果
  print_matrix("Input", h_input, M, N);
  print_matrix("Kernel", h_kernel, K, K);
  print_matrix("CPU Output", h_output_cpu, out_M, out_N);
  print_matrix("GPU Output", h_output_gpu, out_M, out_N);

  // 验证结果
  bool pass = true;
  for (int i = 0; i < out_M*out_N; ++i) {
    if (fabs(h_output_cpu[i] - h_output_gpu[i]) > 1e-3) {
      printf("Mismatch at (%d,%d): CPU=%.4f vs GPU=%.4f\n",
             i/out_N, i%out_N, h_output_cpu[i], h_output_gpu[i]);
      pass = false;
      break;
    }
  }
  printf("Test: %s\n\n", pass ? "PASS" : "FAIL");

  // 清理
  delete[] h_output_cpu;
  delete[] h_output_gpu;
  cudaFree(d_input);
  cudaFree(d_kernel);
  cudaFree(d_output);
}

int main() {
  // 测试用例1：最小可运行测试
  {
    const int M = 4, N = 4, K = 2;
    float h_input[] = {
      1,2,3,4,
      5,6,7,8,
      9,10,11,12,
      13,14,15,16
    };
    float h_kernel[] = {1,0,0,1}; // 2x2核
    printf("==== Test Case 1: Basic Test ====\n");
    test_case(h_input, h_kernel, M, N, K);
  }

  // 测试用例2：边界块测试
  {
    const int M = 34, N = 34, K = 3; // BLOCKSIZE=32, O_Total=30
    float *h_input = new float[M*N];
    float *h_kernel = new float[K*K];

    // 生成渐变数据
    for (int i = 0; i < M*N; ++i) h_input[i] = i % 10;
    for (int i = 0; i < K*K; ++i) h_kernel[i] = (i == K*K/2) ? 1.0f : 0.0f;

    printf("\n==== Test Case 2: Block Boundary Test ====\n");
    test_case(h_input, h_kernel, M, N, K);

    delete[] h_input;
    delete[] h_kernel;
  }

  return 0;
}

In [ ]:
!nvcc conv2d_tiled.cu -o conv2d_tiled -arch=sm_75 && ./conv2d_tiled

## 08 - LayerNorm

考察频率：低。

本质就是行规约。3 阶段：
1. 求 mean：$\mu = \frac{1}{K}\sum x_k$
2. 求 variance + 倒数：$\sigma = \mathrm{rsqrtf}(\frac{1}{K}\sum (x_k - \mu)^2 + \epsilon)$
3. elementwise：$y = a \cdot (x - \mu) \cdot \sigma + b$

reduce 部分套用 warp shuffle 那套就行。

In [ ]:
%%writefile layer_norm.cu
#include <stdio.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <cmath>

#define WARPSIZE 32
#define CEIL(a, b) ((a + b - 1) / b)

__global__ void layerNormKernel(const float *A, float *B, int N, int K,
                              float epsilon, float a, float b) {
  int row = blockIdx.x;
  int lane_id = threadIdx.x % WARPSIZE;
  int iteration = CEIL(K, WARPSIZE);

  __shared__ float s_mean;
  __shared__ float s_variance;

  // Stage 1: 计算均值
  float sum_val = 0;
  for (int i = 0; i < iteration; i++){
    int col = i * WARPSIZE + lane_id;
    if (col < K) sum_val += A[row * K + col];
  }
  for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
    sum_val += __shfl_down_sync(0xFFFFFFFF, sum_val, offset);
  }
  if (lane_id == 0) s_mean = sum_val / K;
  __syncthreads();

  // Stage 2: 计算方差
  float var_sum = 0;
  for (int i = 0; i < iteration; i++) {
    int col = i * WARPSIZE + lane_id;
    if (col < K){
      float val = A[row * K + col];
      var_sum += (val - s_mean) * (val - s_mean);
    }
  }
  for (int offset = WARPSIZE >> 1; offset > 0; offset >>= 1) {
    var_sum += __shfl_down_sync(0xFFFFFFFF, var_sum, offset);
  }
  if (lane_id == 0) s_variance = rsqrtf(var_sum / K + epsilon);
  __syncthreads();

  // Stage 3: 归一化
  for (int i = 0; i < iteration; i++){
    int col = i * WARPSIZE + lane_id;
    if (col < K){
      float val = A[row * K + col];
      float tmp = (val - s_mean) * s_variance;
      B[row * K + col] = a * tmp + b;
    }
  }
}

void layerNorm(const float *A, float *B, int N, int K,
             float epsilon=1e-5f, float a=1.0f, float b=0.0f) {
  dim3 threadPerBlock(WARPSIZE);
  dim3 blockPerGrid(N);
  layerNormKernel<<<blockPerGrid, threadPerBlock>>>(A, B, N, K, epsilon, a, b);
  cudaDeviceSynchronize();
}

// CPU参考实现
void cpu_layer_norm(const float* input, float* output, int N, int K,
                  float epsilon, float gamma, float beta) {
  for (int i = 0; i < N; ++i) {
    // 计算均值
    float mean = 0;
    for (int j = 0; j < K; ++j) {
      mean += input[i*K + j];
    }
    mean /= K;

    // 计算方差
    float variance = 0;
    for (int j = 0; j < K; ++j) {
      float diff = input[i*K + j] - mean;
      variance += diff * diff;
    }
    variance = 1.0f / sqrt(variance / K + epsilon);

    // 归一化
    for (int j = 0; j < K; ++j) {
      output[i*K + j] = gamma * (input[i*K + j] - mean) * variance + beta;
    }
  }
}

void print_vector(const char* name, const float* data, int size) {
  printf("%s (first 5 elements): [ ", name);
  for (int i = 0; i < std::min(5, size); ++i) {
    printf("%.4f ", data[i]);
  }
  printf(size > 5 ? "... ]\n" : "]\n");
}

void test_layer_norm_case(const float* h_input, int N, int K) {
  const int total = N * K;
  float *h_output_cpu = new float[total];
  float *h_output_gpu = new float[total];
  float *d_input, *d_output;

  // CPU计算（使用默认参数gamma=1, beta=0）
  cpu_layer_norm(h_input, h_output_cpu, N, K, 1e-5f, 1.0f, 0.0f);

  // GPU计算
  cudaMalloc(&d_input, total * sizeof(float));
  cudaMalloc(&d_output, total * sizeof(float));
  cudaMemcpy(d_input, h_input, total * sizeof(float), cudaMemcpyHostToDevice);

  layerNorm(d_input, d_output, N, K);

  cudaMemcpy(h_output_gpu, d_output, total * sizeof(float), cudaMemcpyDeviceToHost);

  // 打印结果
  print_vector("Input", h_input, total);
  print_vector("CPU Output", h_output_cpu, total);
  print_vector("GPU Output", h_output_gpu, total);

  // 验证结果
  bool pass = true;
  for (int i = 0; i < total; ++i) {
    if (fabs(h_output_cpu[i] - h_output_gpu[i]) > 1e-4) {
      printf("Mismatch at %d: CPU=%.4f vs GPU=%.4f\n",
            i, h_output_cpu[i], h_output_gpu[i]);
      pass = false;
      break;
    }
  }
  printf("Test: %s\n\n", pass ? "PASS" : "FAIL");

  // 清理
  delete[] h_output_cpu;
  delete[] h_output_gpu;
  cudaFree(d_input);
  cudaFree(d_output);
}

int main() {
  // 测试用例1：小规模测试
  {
    const int N = 2, K = 5;
    float h_input[N][K] = {
      {1.0f, 2.0f, 3.0f, 4.0f, 5.0f},
      {5.0f, 6.0f, 7.0f, 8.0f, 9.0f}
    };
    printf("==== Test Case 1: Small Vector ====\n");
    test_layer_norm_case((float*)h_input, N, K);
  }

  // 测试用例2：随机数据测试
  {
    const int N = 100, K = 256;
    float *h_input = new float[N*K];
    srand(42);
    for (int i = 0; i < N*K; ++i) {
      h_input[i] = (float)rand() / RAND_MAX * 10.0f;
    }
    printf("\n==== Test Case 2: Random Data ====\n");
    test_layer_norm_case(h_input, N, K);
    delete[] h_input;
  }

  return 0;
}

In [ ]:
!nvcc layer_norm.cu -o layer_norm -arch=sm_75 && ./layer_norm